# CAN I BUILD A SNOWMAN?

#### This program assumes previous steps have been completed (see README file).  

#### This program reads in the weather station data files, analyzes the data, creates time series models for max temperature and snow depth for each weather station, and writes out a file containing historical and predicted values for whether or not a snowman could be built in the area around each weather station on a given date. 

In [14]:
# needed for file operations
import os

# needed for data manipulation
import pandas as pd

# needed for time series analysis and forecasting
from statsmodels.tsa.arima.model import ARIMA

# some weather stations have data issues that make ARIMA unhappy, 
# and there are too many weather stations to manage individually,
# so we use this to skip the problematic weather stations
import warnings
warnings.filterwarnings('error')

## VARIABLES

In [ ]:
# SET THE LATITUDE / LONGITUDE BOX FOR THE REGION OF INTEREST 

# NEWBERRY, MI
NORTHMOST = 46.5
SOUTHMOST = 46.0 
EASTMOST = -85.0
WESTMOST = -86.0

# IOWA
#NORTHMOST = 44.0
#SOUTHMOST = 40.0 
#EASTMOST = -89.0
#WESTMOST = -97.0

# continental US
#NORTHMOST = 49.345786
#SOUTHMOST = 24.396308  
#WESTMOST = -125.000000
#EASTMOST = -66.93457

In [ ]:
# CONDITIONS FOR SNOWMAN
MIN_SNOW_DEPTH_FOR_SNOWMAN_IN_INCHES = 4
MIN_TEMP_FOR_SNOWMAN_IN_FAHRENHEIT = 30
MAX_TEMP_FOR_SNOWMAN_IN_FAHRENHEIT = 40

In [ ]:
DIR_RAW_DATA_FILES = 'RawData'
DIR_CLEANED_DATA_FILES = 'CleanedData'
DIR_AGGREGATED_DATA_FILES = 'AggregatedData'
DIR_FORECASTED_DATA_FILES = 'ForecastData'
DIR_FORECASTED_DAILY_DATA_FILES = 'ForecastDailyData'

FILE_STATION_LAT_LONG = 'StationLatLong.csv'
FILE_STATION_LIST = 'StationList.txt'
FILE_FOR_GEOPLOTTING = 'SnowmanData.csv'

ACTUAL_END_DATE = '2024-12-31'
FORECAST_START_DATE = '2025-01-01'
FORECAST_END_DATE = '2039-12-31'

BEGIN_DATE_CUTOFF = '1900-01-01'
END_DATE_CUTOFF = '2039-12-31'

## FUNCTIONS

In [18]:
# This function compiles a list of the weather stations and their coordinates.

def get_station_lat_long_data():
    list_of_all_files = os.listdir(DIR_RAW_DATA_FILES)
    df_station_lat_long = pd.DataFrame(columns=['STATION', 'LATITUDE', 'LONGITUDE'])

    # iterate over the raw files, pulling the station name and coordinates from the first row of each file
    i = 0
    for file in list_of_all_files:
        i += 1
        if i % 1000 == 0:
            print(f'Processing file {i} of {len(list_of_all_files)}')
        file_path = os.path.join(DIR_RAW_DATA_FILES, file)
        df = pd.read_csv(file_path, low_memory=False)
        station = df.iloc[0]['STATION']
        latitude = df.iloc[0]['LATITUDE']
        longitude = df.iloc[0]['LONGITUDE']
        df_station_lat_long.loc[len(df_station_lat_long)] = [station, latitude, longitude]

    # write the dataframe to a csv file
    df_station_lat_long.to_csv(FILE_STATION_LAT_LONG, index=False)

    # finished
    print(FILE_STATION_LAT_LONG + ' file created.')


In [19]:
# This function reads in the station latitudes and longitudes, 
# filters them by the lat/long box, 
# and writes the results to a file.

def get_stations_in_latlong_box():
    # read in the file with station latitudes and longitudes
    df_station_lat_long = pd.read_csv(FILE_STATION_LAT_LONG)

    # filter the stations to only include those within the lat/long box
    df_station_lat_long = df_station_lat_long[(df_station_lat_long['LATITUDE'] <= NORTHMOST) & 
                                              (df_station_lat_long['LATITUDE'] >= SOUTHMOST) & 
                                              (df_station_lat_long['LONGITUDE'] <= EASTMOST) & 
                                              (df_station_lat_long['LONGITUDE'] >= WESTMOST)]

    # write the stations to a list
    stations = df_station_lat_long['STATION'].tolist()

    # write the station list to a file
    with open(FILE_STATION_LIST, 'w') as f:
        for station in stations:
            f.write(station + '\n')
    
    # finished
    print(FILE_STATION_LIST + ' file created.')        

In [20]:
# This function reads in the list of weather stations, 
# does additional filtering to only keep the weather stations that meet our requirements, 
# cleans the data, and writes a new file for each weather station.

def filter_and_clean_data():
    # get the list of weather stations that are within my lat/long box (output of previous step)
    ws_file_list = []
    with open(FILE_STATION_LIST, 'r') as file:
        for line in file:
            ws_file_list.append(line.strip())

    # list of the columns we need for our analysis
    columns_we_need = ['DATE', 'SNWD', 'TMAX']

    # loop through the list of weather stations
    i = 0
    for file in ws_file_list:
        i += 1
        if i % 100 == 0:
            print(f'Processing file {i} of {len(ws_file_list)}')

        # read the data from the weather station file
        ws_df = pd.read_csv(os.path.join(DIR_RAW_DATA_FILES, file+".csv"), low_memory=False)
    
        # only include the weather stations that have the columns we need
        if set(ws_df.columns).issuperset(columns_we_need):
            # filter the data to only include date, snow depth, and max temperature
            ws_df = ws_df[['DATE', 'SNWD', 'TMAX']].dropna()
            ws_df['DATE'] = pd.to_datetime(ws_df['DATE'])
            ws_df['TMAX'] = pd.to_numeric(ws_df['TMAX'], errors='coerce')
            ws_df['SNWD'] = pd.to_numeric(ws_df['SNWD'], errors='coerce')

            # in raw data, temp is in tenths of celsius, 
            # convert to fahrenheit
            ws_df['TMAX'] = ws_df['TMAX'] / 10 * 9/5 + 32
            
            # in raw data, snow depth is in millimeters, 
            # convert to inches
            ws_df['SNWD'] = ws_df['SNWD'] / 25.4
        
            # only keep weather stations that have at least a few years of data
            if len(ws_df) > 1200:
                # only keep the weather stations that have data up to current year
                last_date = ws_df.iloc[len(ws_df)-1]['DATE']
                if last_date.strftime('%Y-%m-%d') >= ACTUAL_END_DATE:
                    # cut data off so all weather stations have the same end date
                    ws_df = ws_df[ws_df['DATE'] <= ACTUAL_END_DATE]
                    
                    # save the filtered data to a new file
                    ws_df.to_csv(os.path.join(DIR_CLEANED_DATA_FILES, file+".csv"), index=False)

    # finished
    print('Data filtered and cleaned.')       

    

In [21]:
# This function aggregates the daily observations to monthly averages for each weather station
def aggregate_data():

    # get all files in the directory
    list_of_all_files = os.listdir(DIR_CLEANED_DATA_FILES)
    
    # iterate over all files
    i = 0
    for file in list_of_all_files:
        i += 1
        if i % 100 == 0:
            print(f'Processing file {i} of {len(list_of_all_files)}')

        file_path = os.path.join(DIR_CLEANED_DATA_FILES, file)
        df = pd.read_csv(file_path, low_memory=False)
        df['DATE'] = pd.to_datetime(df['DATE'])
        df['TMAX'] = pd.to_numeric(df['TMAX'], errors='coerce')
        df['SNWD'] = pd.to_numeric(df['SNWD'], errors='coerce')
        df.set_index('DATE', inplace=True)

        # resample the data to monthly averages
        df_date_tmax_snwd_agg = df.resample('MS').mean()

        # interpolate missing monthly values (not often applicable, but covers some edge cases)
        df_date_tmax_snwd_agg = df_date_tmax_snwd_agg.interpolate()

        # write to file
        file_path = os.path.join(DIR_AGGREGATED_DATA_FILES, file)
        df_date_tmax_snwd_agg.to_csv(file_path)

    # finished
    print('Data aggregated.')               

In [22]:
# This function iterates over all the aggregated data files, 
# forecasts the monthly averages, 
# and writes the results to new files.
def make_monthly_predictions():
    # get all files in the directory
    list_of_all_files = os.listdir(DIR_AGGREGATED_DATA_FILES)
    
    # iterate over all files
    i = 0
    for file in list_of_all_files:
        i += 1
        if i % 100 == 0:
            print(f'Processing file {i} of {len(list_of_all_files)}')
        file_path = os.path.join(DIR_AGGREGATED_DATA_FILES, file)

        # read the historical data from the weather station file
        ws_df = pd.read_csv(os.path.join(DIR_AGGREGATED_DATA_FILES, file), low_memory=False, parse_dates=['DATE'])

        # do temp prediction
        temp_df = ws_df[['DATE','TMAX']].copy().dropna()
        temp_df = temp_df.set_index('DATE')
        temp_df.index = pd.DatetimeIndex(temp_df.index).to_period('M')
        try:
            model = ARIMA(temp_df, order=(1,0,1), seasonal_order=(0,1,1,12)).fit()
        except:
            # if the model fails to fit, skip this weather station 
            continue  
        temp_forecast = model.predict(start=FORECAST_START_DATE, end=FORECAST_END_DATE)
        temp_forecast.rename('TMAX', inplace=True)
        temp_all = pd.concat([temp_df, temp_forecast]).rename(columns={0:'TMAX'})

        # do snow depth prediction
        snwd_df = ws_df[['DATE','SNWD']].copy().dropna()
        snwd_df = snwd_df.set_index('DATE')
        snwd_df.index = pd.DatetimeIndex(snwd_df.index).to_period('M')
        try:
            model = ARIMA(snwd_df, order=(2,0,0), seasonal_order=(2,1,1,12)).fit()
        except:
            # if the model fails to fit, skip this weather station 
            continue  
        snwd_forecast = model.predict(start=FORECAST_START_DATE, end=FORECAST_END_DATE)
        snwd_forecast.rename('SNWD', inplace=True)
        snwd_all = pd.concat([snwd_df, snwd_forecast]).rename(columns={0:'SNWD'})

        output_df = pd.concat([temp_all, snwd_all], axis=1).reset_index().rename(columns={'index':'DATE'}).sort_values(by='DATE')

        output_file_path = os.path.join(DIR_FORECASTED_DATA_FILES, file)
        output_df.to_csv(output_file_path, index=False)

    # finished
    print('Monthly predictions finished.')       

In [23]:
# This function infers daily predictions from monthly predictions
# and writes the results to new files.
def infer_daily_predictions():
    # get all files in the directory
    list_of_all_files = os.listdir(DIR_FORECASTED_DATA_FILES)

    # for each file in the forecast folder
    i = 0
    for file in list_of_all_files:
        i += 1
        if i % 100 == 0:
            print(f'Processing file {i} of {len(list_of_all_files)}')

        file_path = os.path.join(DIR_FORECASTED_DATA_FILES, file)
        forecast_df = pd.read_csv(file_path, low_memory=False, parse_dates=['DATE'])
        forecast_df = forecast_df[forecast_df['DATE'] > ACTUAL_END_DATE]
        forecast_df.set_index('DATE', inplace=True)
        forecast_df = forecast_df.resample('D').asfreq()

        file_path = os.path.join(DIR_CLEANED_DATA_FILES, file)
        cleaned_df = pd.read_csv(file_path, low_memory=False, parse_dates=['DATE'])
        cleaned_df = cleaned_df[cleaned_df['DATE'] <= ACTUAL_END_DATE]
        cleaned_df = cleaned_df[['DATE', 'SNWD', 'TMAX']]
        cleaned_df.set_index('DATE', inplace=True)

        output_df = pd.concat([cleaned_df, forecast_df])
        output_df['TMAX'] = output_df['TMAX'].interpolate()
        output_df['SNWD'] = output_df['SNWD'].interpolate()
        output_df.reset_index(inplace=True)

        # where snow depth is negative, set it to zero
        output_df.loc[output_df['SNWD'] < 0, 'SNWD'] = 0

        # set snowman flag
        output_df['SNOWMAN'] = 0
        output_df.loc[(output_df['SNWD'] > MIN_SNOW_DEPTH_FOR_SNOWMAN_IN_INCHES) & 
                      (output_df['TMAX'] >= MIN_TEMP_FOR_SNOWMAN_IN_FAHRENHEIT) & 
                      (output_df['TMAX'] <= MAX_TEMP_FOR_SNOWMAN_IN_FAHRENHEIT), 
                      'SNOWMAN'] = 1
    
        output_file_path = os.path.join(DIR_FORECASTED_DAILY_DATA_FILES, file)
        output_df.to_csv(output_file_path, index=False)

    # finished
    print('Daily predictions finished.')   

In [24]:
# This function prepares the data for geoplotting by reading in the forecasted daily data files,
# joining with the station lat/long data,
# creating a dataframe with date, snowman flag, latitude, and longitude,
# and writing the results to a new file.
def prep_for_geoplotting():
    # get station lat/long data
    station_lat_long_df = pd.read_csv(FILE_STATION_LAT_LONG)
    
    # get all files in the forecasted daily data directory
    list_of_all_files = os.listdir(DIR_FORECASTED_DAILY_DATA_FILES)

    # snowman_df will store the snowman data for all stations
    snowman_df = pd.DataFrame()

    # for each file in the forecasted daily data folder
    i = 0
    for file in list_of_all_files:
        i += 1
        if i % 100 == 0:
            print(f'Processing file {i} of {len(list_of_all_files)}')

        # read this station's date/snowman data
        file_path = os.path.join(DIR_FORECASTED_DAILY_DATA_FILES, file)
        ws_df = pd.read_csv(file_path, low_memory=False, parse_dates=['DATE'])
        ws_df = ws_df[['DATE', 'SNOWMAN']]
    
        # get the station's latitude and longitude from the station lat/long data
        station = file.split(".")[0]
        latitude = station_lat_long_df[station_lat_long_df['STATION'] == station]['LATITUDE'].values[0]
        longitude = station_lat_long_df[station_lat_long_df['STATION'] == station]['LONGITUDE'].values[0]
        ws_df['LATITUDE'] = latitude
        ws_df['LONGITUDE'] = longitude

        # due to big data issues, need to cutoff some data
        ws_df = ws_df[ws_df['DATE'] >= BEGIN_DATE_CUTOFF]
        ws_df = ws_df[ws_df['DATE'] <= END_DATE_CUTOFF]

        # append this station's data to the snowman_df
        snowman_df = pd.concat([snowman_df, ws_df])

    # save the snowman data to a file
    snowman_df.to_csv(FILE_FOR_GEOPLOTTING, index=False)

    # finished
    print(FILE_FOR_GEOPLOTTING + ' finished.')   

## RUN

In [ ]:
# Compile a list of the weather stations and their coordinates
# to assist in filtering the data geographically, 
get_station_lat_long_data()

In [ ]:
# Compile a list of the weather stations that fall within the lat/long box
get_stations_in_latlong_box()

In [27]:
# Filter out the weather stations that do not meet our requirements.
# For each weather station, 
# clean the data, and write the cleaned data to a new file
filter_and_clean_data()

Processing file 100 of 78956
Processing file 200 of 78956
Processing file 300 of 78956
Processing file 400 of 78956
Processing file 500 of 78956
Processing file 600 of 78956
Processing file 700 of 78956
Processing file 800 of 78956
Processing file 900 of 78956
Processing file 1000 of 78956
Processing file 1100 of 78956
Processing file 1200 of 78956
Processing file 1300 of 78956
Processing file 1400 of 78956
Processing file 1500 of 78956
Processing file 1600 of 78956
Processing file 1700 of 78956
Processing file 1800 of 78956
Processing file 1900 of 78956
Processing file 2000 of 78956
Processing file 2100 of 78956
Processing file 2200 of 78956
Processing file 2300 of 78956
Processing file 2400 of 78956
Processing file 2500 of 78956
Processing file 2600 of 78956
Processing file 2700 of 78956
Processing file 2800 of 78956
Processing file 2900 of 78956
Processing file 3000 of 78956
Processing file 3100 of 78956
Processing file 3200 of 78956
Processing file 3300 of 78956
Processing file 340

In [28]:
# Aggregate the daily observations to monthly averages for each weather station
aggregate_data()

Processing file 100 of 4986
Processing file 200 of 4986
Processing file 300 of 4986
Processing file 400 of 4986
Processing file 500 of 4986
Processing file 600 of 4986
Processing file 700 of 4986
Processing file 800 of 4986
Processing file 900 of 4986
Processing file 1000 of 4986
Processing file 1100 of 4986
Processing file 1200 of 4986
Processing file 1300 of 4986
Processing file 1400 of 4986
Processing file 1500 of 4986
Processing file 1600 of 4986
Processing file 1700 of 4986
Processing file 1800 of 4986
Processing file 1900 of 4986
Processing file 2000 of 4986
Processing file 2100 of 4986
Processing file 2200 of 4986
Processing file 2300 of 4986
Processing file 2400 of 4986
Processing file 2500 of 4986
Processing file 2600 of 4986
Processing file 2700 of 4986
Processing file 2800 of 4986
Processing file 2900 of 4986
Processing file 3000 of 4986
Processing file 3100 of 4986
Processing file 3200 of 4986
Processing file 3300 of 4986
Processing file 3400 of 4986
Processing file 3500 of

In [29]:
# Now that we have the monthly historical data ready for analysis,
# we can proceed with building the models and making predictions.
make_monthly_predictions()

Processing file 100 of 4986
Processing file 200 of 4986
Processing file 300 of 4986
Processing file 400 of 4986
Processing file 500 of 4986
Processing file 600 of 4986
Processing file 700 of 4986
Processing file 800 of 4986
Processing file 900 of 4986
Processing file 1000 of 4986
Processing file 1100 of 4986
Processing file 1200 of 4986
Processing file 1300 of 4986
Processing file 1400 of 4986
Processing file 1500 of 4986
Processing file 1600 of 4986
Processing file 1700 of 4986
Processing file 1800 of 4986
Processing file 1900 of 4986
Processing file 2000 of 4986
Processing file 2100 of 4986
Processing file 2200 of 4986
Processing file 2300 of 4986
Processing file 2400 of 4986
Processing file 2500 of 4986
Processing file 2600 of 4986
Processing file 2700 of 4986
Processing file 2800 of 4986
Processing file 2900 of 4986
Processing file 3000 of 4986
Processing file 3100 of 4986
Processing file 3200 of 4986
Processing file 3300 of 4986
Processing file 3400 of 4986
Processing file 3500 of

In [30]:
# With our monthly predictions in hand,
# we can now infer the daily predictions
infer_daily_predictions()

Processing file 100 of 4299
Processing file 200 of 4299
Processing file 300 of 4299
Processing file 400 of 4299
Processing file 500 of 4299
Processing file 600 of 4299
Processing file 700 of 4299
Processing file 800 of 4299
Processing file 900 of 4299
Processing file 1000 of 4299
Processing file 1100 of 4299
Processing file 1200 of 4299
Processing file 1300 of 4299
Processing file 1400 of 4299
Processing file 1500 of 4299
Processing file 1600 of 4299
Processing file 1700 of 4299
Processing file 1800 of 4299
Processing file 1900 of 4299
Processing file 2000 of 4299
Processing file 2100 of 4299
Processing file 2200 of 4299
Processing file 2300 of 4299
Processing file 2400 of 4299
Processing file 2500 of 4299
Processing file 2600 of 4299
Processing file 2700 of 4299
Processing file 2800 of 4299
Processing file 2900 of 4299
Processing file 3000 of 4299
Processing file 3100 of 4299
Processing file 3200 of 4299
Processing file 3300 of 4299
Processing file 3400 of 4299
Processing file 3500 of

In [31]:
# Prepare the file used for the geoplotting script
prep_for_geoplotting()

Processing file 100 of 4299
Processing file 200 of 4299
Processing file 300 of 4299
Processing file 400 of 4299
Processing file 500 of 4299
Processing file 600 of 4299
Processing file 700 of 4299
Processing file 800 of 4299
Processing file 900 of 4299
Processing file 1000 of 4299
Processing file 1100 of 4299
Processing file 1200 of 4299
Processing file 1300 of 4299
Processing file 1400 of 4299
Processing file 1500 of 4299
Processing file 1600 of 4299
Processing file 1700 of 4299
Processing file 1800 of 4299
Processing file 1900 of 4299
Processing file 2000 of 4299
Processing file 2100 of 4299
Processing file 2200 of 4299
Processing file 2300 of 4299
Processing file 2400 of 4299
Processing file 2500 of 4299
Processing file 2600 of 4299
Processing file 2700 of 4299
Processing file 2800 of 4299
Processing file 2900 of 4299
Processing file 3000 of 4299
Processing file 3100 of 4299
Processing file 3200 of 4299
Processing file 3300 of 4299
Processing file 3400 of 4299
Processing file 3500 of